# IPL Match & Player Performance Analysis

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

## Project Objective
This project analyzes IPL match and ball-by-ball data to understand team performance, player performance, toss outcomes, scoring patterns, and venue trends.

> **Dataset required:** `matches.csv` and `deliveries.csv` from a standard IPL match/ball-by-ball dataset.  
> Upload both CSV files when running this notebook in Google Colab.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully")


## 1. Load the IPL Dataset

In [ ]:
# Upload matches.csv and deliveries.csv in Google Colab
from google.colab import files
uploaded = files.upload()

matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)


In [ ]:
# First look at the data
display(matches.head())
display(deliveries.head())

print("\nMatches columns:")
print(matches.columns.tolist())

print("\nDeliveries columns:")
print(deliveries.columns.tolist())


## 2. Data Cleaning and Preparation

In [ ]:
# Basic cleaning
matches = matches.drop_duplicates().copy()
deliveries = deliveries.drop_duplicates().copy()

# Convert date column when available
if 'date' in matches.columns:
    matches['date'] = pd.to_datetime(matches['date'], errors='coerce')

print("Missing values in matches:")
display(matches.isnull().sum().sort_values(ascending=False).head(15))

print("\nMissing values in deliveries:")
display(deliveries.isnull().sum().sort_values(ascending=False).head(15))


## 3. Season and Match Analysis

In [ ]:
# Number of matches by season
if 'season' in matches.columns:
    season_matches = matches.groupby('season').size().reset_index(name='matches')
    display(season_matches)

    plt.figure(figsize=(10,5))
    sns.lineplot(data=season_matches, x='season', y='matches', marker='o')
    plt.title('Number of IPL Matches by Season')
    plt.xlabel('Season')
    plt.ylabel('Number of Matches')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 4. Team Performance

In [ ]:
# Count wins by team
if 'winner' in matches.columns:
    team_wins = matches['winner'].value_counts().reset_index()
    team_wins.columns = ['team', 'wins']

    display(team_wins)

    plt.figure(figsize=(10,6))
    sns.barplot(data=team_wins.head(15), x='wins', y='team')
    plt.title('Teams with Most Match Wins')
    plt.xlabel('Wins')
    plt.ylabel('Team')
    plt.tight_layout()
    plt.show()


## 5. Toss Analysis

In [ ]:
# Toss decision analysis
if 'toss_decision' in matches.columns:
    toss_decision = matches['toss_decision'].value_counts().reset_index()
    toss_decision.columns = ['decision', 'count']
    display(toss_decision)

    plt.figure(figsize=(6,5))
    sns.barplot(data=toss_decision, x='decision', y='count')
    plt.title('Toss Decision Distribution')
    plt.xlabel('Toss Decision')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

# Toss winner vs match winner
if {'toss_winner', 'winner'}.issubset(matches.columns):
    matches['toss_winner_won'] = matches['toss_winner'] == matches['winner']
    toss_result = matches['toss_winner_won'].value_counts().rename(index={
        True: 'Toss winner also won',
        False: 'Toss winner lost'
    }).reset_index()
    toss_result.columns = ['result', 'matches']
    display(toss_result)


## 6. Player Batting Performance

In [ ]:
# Detect common batting columns used in IPL datasets
if {'batter', 'batsman_runs'}.issubset(deliveries.columns):
    batter_runs = (
        deliveries.groupby('batter')['batsman_runs']
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    batter_runs.columns = ['player', 'runs']

    display(batter_runs.head(15))

    plt.figure(figsize=(10,7))
    sns.barplot(data=batter_runs.head(10), x='runs', y='player')
    plt.title('Top Run Scorers')
    plt.xlabel('Runs')
    plt.ylabel('Player')
    plt.tight_layout()
    plt.show()


## 7. Strike Rate Analysis

In [ ]:
if {'batter', 'batsman_runs'}.issubset(deliveries.columns):
    batting_stats = (
        deliveries.groupby('batter')
        .agg(
            runs=('batsman_runs', 'sum'),
            balls=('batsman_runs', 'size')
        )
        .reset_index()
    )

    # Consider players with at least 100 balls for a more meaningful comparison
    batting_stats = batting_stats[batting_stats['balls'] >= 100].copy()
    batting_stats['strike_rate'] = (
        batting_stats['runs'] / batting_stats['balls'] * 100
    )

    display(batting_stats.sort_values('strike_rate', ascending=False).head(15))


## 8. Bowling Performance

In [ ]:
# Wicket analysis using common IPL delivery columns
if {'bowler', 'is_wicket'}.issubset(deliveries.columns):
    wicket_events = deliveries[deliveries['is_wicket'] == 1].copy()

    # Exclude run-outs when dismissal information is available
    if 'dismissal_kind' in wicket_events.columns:
        wicket_events = wicket_events[
            ~wicket_events['dismissal_kind'].isin(['run out', 'retired hurt', 'obstructing the field'])
        ]

    wickets = wicket_events['bowler'].value_counts().reset_index()
    wickets.columns = ['bowler', 'wickets']

    display(wickets.head(15))

    plt.figure(figsize=(10,7))
    sns.barplot(data=wickets.head(10), x='wickets', y='bowler')
    plt.title('Top Wicket Takers')
    plt.xlabel('Wickets')
    plt.ylabel('Bowler')
    plt.tight_layout()
    plt.show()


## 9. Team Run Analysis

In [ ]:
if {'batting_team', 'total_runs'}.issubset(deliveries.columns):
    team_runs = (
        deliveries.groupby('batting_team')['total_runs']
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    team_runs.columns = ['team', 'total_runs']

    display(team_runs)

    plt.figure(figsize=(10,6))
    sns.barplot(data=team_runs, x='total_runs', y='team')
    plt.title('Total Runs Scored by Team')
    plt.xlabel('Total Runs')
    plt.ylabel('Team')
    plt.tight_layout()
    plt.show()


## 10. Venue Analysis

In [ ]:
if 'venue' in matches.columns:
    venue_matches = matches['venue'].value_counts().reset_index()
    venue_matches.columns = ['venue', 'matches']

    display(venue_matches.head(15))

    plt.figure(figsize=(10,7))
    sns.barplot(data=venue_matches.head(10), x='matches', y='venue')
    plt.title('Venues with Most IPL Matches')
    plt.xlabel('Matches')
    plt.ylabel('Venue')
    plt.tight_layout()
    plt.show()


## 11. Score Distribution

In [ ]:
if {'match_id', 'inning', 'total_runs'}.issubset(deliveries.columns):
    innings_score = (
        deliveries.groupby(['match_id', 'inning'])['total_runs']
        .sum()
        .reset_index()
    )

    display(innings_score.head())

    plt.figure(figsize=(9,5))
    sns.histplot(innings_score['total_runs'], bins=30, kde=True)
    plt.title('Distribution of Innings Scores')
    plt.xlabel('Innings Score')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()


## 12. Advanced Pandas Analysis

In [ ]:
# Example: rank teams by total runs
if 'team_runs' in globals():
    team_runs['rank'] = team_runs['total_runs'].rank(
        method='dense', ascending=False
    ).astype(int)

    display(team_runs.sort_values('rank').head(15))

# Example: pivot table for season-wise team wins
if {'season', 'winner'}.issubset(matches.columns):
    season_team_wins = pd.pivot_table(
        matches,
        index='season',
        columns='winner',
        values='id' if 'id' in matches.columns else matches.columns[0],
        aggfunc='count',
        fill_value=0
    )
    display(season_team_wins)


## 13. Key Insights

The exact values below are generated from the uploaded dataset, so they should be reviewed after running the notebook. This section avoids hard-coded claims and can be used to write the final project findings.

- Team performance was compared using total wins and total runs.
- Toss decisions and the relationship between toss winners and match winners were analyzed.
- Player performance was studied using total runs, strike rate, and wickets.
- Venue-level match counts were analyzed to understand where IPL matches were played most frequently.
- Innings score distribution was used to understand scoring patterns.
- Advanced Pandas operations such as GroupBy, Pivot Table, Ranking, filtering, and aggregation were used throughout the analysis.


## Resume Project Description

### IPL Match & Player Performance Analysis | Python, Pandas, NumPy, Matplotlib
- Analyzed IPL match and ball-by-ball data using Python, Pandas, NumPy, and Matplotlib to study team and player performance trends.
- Performed data cleaning, transformation, aggregation, and exploratory data analysis on match and player-level data.
- Analyzed team wins, toss outcomes, player runs, strike rates, wickets, scoring patterns, and venue-level trends.
- Used advanced Pandas operations including GroupBy, Merge, Pivot Tables, Ranking, filtering, and aggregations for performance analysis.
- Created charts to compare team performance, player statistics, scoring patterns, and venue trends.
- Derived data-driven insights from IPL statistics and presented findings through clear tables and visualizations.
